# Restore cached KITTI inputs for the next experiment
Latest revision: **A2-LITE-INPUT-CACHE-2026-10-14-r1**

Run this in the SAME runtime as the next GPU experiment, before model setup. It does not train a model or use the GPU. CPU preparation must already have produced the reusable cache on Drive.

Copy ONE archive from Drive to local disk, verifying64-MiB chunks and the full archive hash. Resume a verified owned partial transfer when possible; preserve conflicting files. Extract only the expected images, labels, calibration and disjoint Chen split lists; reject extra/unsafe/duplicate members and verify every input's reviewed hash.

Reuse the local archive and correct extracted files within this runtime. A Colab reset loses local data; rerun restore, not the one-time CPU preparation. Do not extract directly from Drive and do not change the experiment's data selection, losses or training budget.

Budget local space for the archive plus unpacked files. Copying/extraction is still CPU/I/O work, so brief GPU idle time is expected. Real startup time depends on Drive and local-disk throughput.

## 1. Mount Drive, set next experiment paths, and define all helpers
Set DATASET to the next experiment's dedicated local dataset folder BEFORE its manifest is created. No current experiment paths are changed here automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, shlex, subprocess, sys
REVISION='A2-LITE-INPUT-CACHE-2026-10-14-r1'
MOBILE_REPO=Path('/content/mobile_adas3d')
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
CACHE=Path('/content/drive/MyDrive/mobile_adas3d_inputs/kitti_chen_a2_v1')
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/cache_a2_lite_inputs.py'
LOGS=CACHE/'operation_logs';LOGS.mkdir(parents=True,exist_ok=True)
def run_logged(command,name,cwd=None):
    command=list(map(str,command));log_path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True);tail=deque(maxlen=40)
    with log_path.open('w') as log:
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; log={log_path}\n'+'\n'.join(tail))
    return log_path
print(REVISION,'\nReusable input cache:',CACHE)
# Replace this with the next experiment's actual DATASET path.
DATASET=Path('/content/kitti_a2_lite_accuracy_next_experiment')
LOCAL_ARCHIVE=Path('/content/a2_lite_archive_local/kitti_chen_7481.zip')
READY=False


## 2. Update main and verify the helper
No GPU libraries or ordinary dependencies need installing. This input preparation can run before the next model-specific environment setup.


In [ ]:
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Pull the published cache workflow from main')
if not PILOT.exists():raise FileNotFoundError('Set PILOT to the original512 folder/ZIP in section1')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_input_cache.py','-v'],'cache_tests',MOBILE_REPO)
if not (CACHE/'kitti_input_cache.json').is_file():raise FileNotFoundError('Run the CPU cache-preparation notebook once first')
READY=True


## 3. Transfer once, extract locally and verify all inputs
The normal experiment prepare stage should still check its own frozen data identity. This helper only makes its input files local in advance.


In [ ]:
if not READY:raise RuntimeError('Run sections1 and2 first')
run_logged([PYTHON,'-u',SCRIPT,'restore','--pilot',PILOT,'--cache',CACHE,'--dataset',DATASET,'--local-archive',LOCAL_ARCHIVE],'restore_input_archive',MOBILE_REPO)
report=json.loads((DATASET/'a2_lite_archive_restore.json').read_text())
assert report['complete'] and report['every_input_hash_verified']
assert report['train_frames']==3712 and report['val_frames']==3769
print(json.dumps(report,indent=2))
print('Ready for the next experiment. Use DATASET =',DATASET)
print('Local split lists =',DATASET/'ImageSets')
print('Continue the model-specific notebook in this SAME runtime; do not select training frames from val.txt.')
